# Phase 3 — evaluate the detector

Measures the trained detector on the **held-out test split**, which nothing has touched: validation drove early stopping and model selection, so validation numbers are mildly optimistic. This is the number that goes in the README.

Three outputs:

1. **Ultralytics mAP** on test — the standard metric.
2. **Slices by plate size** — an aggregate of 0.98 can hide 0.6 on plates under 32 px, and that is exactly the population Phase 4's OCR will fail on.
3. **A failure gallery** — the worst images with predictions drawn on them. Metrics say *that* something is wrong; only the images say what.

**Gate: mAP@50 ≥ 0.85 on test.** Below that, iterate on Phase 1 or 2 rather than pushing on — OCR can never read a plate the detector missed.

In [ ]:
REPO = "https://github.com/fayazhussain2821/Automatic-License-Plate-Recognition.git"
BRANCH = "main"

import os

if os.path.isdir("/content/ALPR"):
    !cd /content/ALPR && git fetch --quiet origin && git checkout --quiet $BRANCH && git pull --quiet
else:
    !git clone --quiet --branch $BRANCH $REPO /content/ALPR

%cd /content/ALPR
!pip install --quiet -e .

import sys

SRC = "/content/ALPR/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import alpr

print(f"alpr {alpr.__version__} importable")

## 0. Make sure the dataset exists

`ensure_dataset()` is a no-op when this session already has the data, and a full rebuild (download → manifest → split → export) when it does not. That makes this notebook runnable on its own in a fresh runtime — which matters, because free Colab allows one session and `/content` does not survive it.

In [ ]:
pip install roboflow

In [ ]:
from alpr.build import ensure_dataset

# Rebuilds the dataset if this session does not already have it, and returns
# immediately if it does. Colab's free tier gives one GPU session and
# everything under /content dies with it, so a notebook that depends on a
# previous notebook's output usually cannot run at all.
#
# Needs ROBOFLOW_API_KEY in the 🔑 Secrets panel only when a download is
# actually required.
DATA = ensure_dataset()
print(f"dataset ready: {DATA}")

## 1. Locate the weights

If you trained in this session, `best_weights` finds them. If the session was lost, upload the `best.pt` you downloaded and point `WEIGHTS` at it.

In [ ]:
from pathlib import Path

from alpr.train import TrainConfig, best_weights

config = TrainConfig.from_yaml("configs/detector.yaml")

try:
    WEIGHTS = best_weights(config)
except Exception as exc:
    print(f"not found in this session ({exc})")
    print("Upload best.pt with the file browser, then set WEIGHTS manually.")
    WEIGHTS = Path("/content/best.pt")

print(f"weights: {WEIGHTS}  ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")

## 2. Ultralytics mAP on the test split

`split="test"` is the important argument. The default is `val`, which would re-report the number training already optimized against.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))
metrics = model.val(data=str(DATA), split="test", imgsz=config.imgsz, verbose=False)

print(f"mAP@50      {metrics.box.map50:.4f}")
print(f"mAP@50-95   {metrics.box.map:.4f}")
print(f"precision   {metrics.box.mp:.4f}")
print(f"recall      {metrics.box.mr:.4f}")
print()
print("GATE: mAP@50 >= 0.85 —", "PASS" if metrics.box.map50 >= 0.85 else "FAIL")

## 3. Run the detector over the test split

Predictions are needed for the size slices and the gallery, neither of which Ultralytics provides.

In [ ]:
from alpr.data import Split, read_manifest, split_records
from alpr.detect import PlateDetector

records = read_manifest("data/manifest.jsonl")
assignment = split_records(records, seed=0)  # same seed as Phase 1
test_records = assignment.partition(records)[Split.TEST]
print(f"{len(test_records)} test images")

detector = PlateDetector(WEIGHTS, imgsz=config.imgsz)
print(f"device: {detector.device}")

RAW = Path("data/raw")
paths = [str(RAW / r.file_name) for r in test_records]

predictions = []
BATCH = 32
for start in range(0, len(paths), BATCH):
    predictions.extend(detector.detect_batch(paths[start : start + BATCH]))
    print(f"  {min(start + BATCH, len(paths))}/{len(paths)}", end="\r")

print(f"\n{sum(len(p) for p in predictions)} detections")

## 4. Sliced metrics

The row to watch is `tiny (<32px)`. If its recall is far below the rest, the ceiling on end-to-end accuracy is set by detection on small plates — and raising `imgsz` in Phase 2 is the fix, not more epochs.

In [ ]:
from alpr.evaluate import evaluate_records

report = evaluate_records(test_records, predictions)
print(report.report())

## 5. Failure gallery

Green = correct, red = missed plate, orange = false positive.

In [ ]:
from IPython.display import Image as ShowImage
from IPython.display import display

from alpr.evaluate import render_failure_gallery

written = render_failure_gallery(
    report, test_records, predictions, RAW, "outputs/failures", limit=20
)
print(f"{len(written)} failure image(s) written")

for path in written[:8]:
    print(path.name)
    display(ShowImage(filename=str(path), width=300))

Look for a pattern rather than counting. Motion blur, night shots, plates at extreme angles and duplicate boxes on one plate all call for different fixes — more data, a different augmentation, or a lower NMS threshold. If the gallery is empty, there is nothing left to learn from failures at this IoU threshold.